# Notebook 02: Exploratory Data Analysis (EDA)
## SAMATRIX ResumeForge 2026 — Multiclass Resume Classification

### Objective
Provide comprehensive empirical analysis of resume properties:
1. Class distribution & imbalance ratios across the 24 categories.
2. Resume length distributions (characters, words, tokens) overall and per category.
3. Most frequent terms and class-specific WordClouds.
4. N-gram analysis (Unigrams, Bigrams, Trigrams).
5. Discriminative vocabulary scoring per profession.


In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

# Robust project root setup (works from notebooks/ or workspace root)
if os.path.exists('src'):
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists('../src'):
    PROJECT_ROOT = os.path.abspath('..')
else:
    PROJECT_ROOT = os.path.abspath('.')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import CSV_PATH, FIGURES_DIR, CATEGORIES
from src.preprocessing.text_cleaner import preprocess_resume

df = pd.read_csv(CSV_PATH)
df['word_count'] = df['Resume_str'].fillna('').str.split().str.len()
df['char_len'] = df['Resume_str'].fillna('').str.len()
print(f"Loaded {len(df)} resumes across {df['Category'].nunique()} categories")


### 1. Class Distribution Analysis


In [ ]:
cat_counts = df['Category'].value_counts()
print("Top 5 categories:")
print(cat_counts.head(5))
print()
print("Bottom 5 categories (minority classes):")
print(cat_counts.tail(5))
imbalance_ratio = cat_counts.max() / cat_counts.min()
print()
print(f"Imbalance Ratio (Max/Min): {imbalance_ratio:.2f}")


In [ ]:
plt.figure(figsize=(14, 6))
sns.barplot(x=cat_counts.values, y=cat_counts.index, hue=cat_counts.index, palette='viridis', legend=False)
plt.title('Resume Count per Category (24 Classes)')
plt.xlabel('Number of Samples')
plt.tight_layout()
plt.show()


### 2. Resume Length Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['word_count'], bins=40, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Word Count Distribution')
axes[0].set_xlabel('Words')

sns.boxplot(x='word_count', y='Category', data=df, ax=axes[1], orient='h', hue='Category', palette='magma', legend=False)
axes[1].set_title('Word Count by Category')
axes[1].set_xlabel('Word Count')
plt.tight_layout()
plt.show()


### 3. Top Discriminative Vocabulary & N-Grams


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF across categories
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X = tfidf.fit_transform(df['Resume_str'].fillna(''))
features = tfidf.get_feature_names_out()

# Display top terms for key classes
for target in ['INFORMATION-TECHNOLOGY', 'HEALTHCARE', 'CHEF', 'FINANCE']:
    mask = (df['Category'] == target).to_numpy()
    cls_mean = np.asarray(X[mask].mean(axis=0)).ravel()
    rest_mean = np.asarray(X[~mask].mean(axis=0)).ravel()
    disc = cls_mean - rest_mean
    top_terms = [features[i] for i in disc.argsort()[-8:][::-1]]
    print(f"Top discriminative terms for {target}: {', '.join(top_terms)}")
